# Replication: Gradient-Based Learning Applied to Document Recognition
**Author:** Belyagoubi Mohammed Abdelilah
**Reference:** Yann LeCun et al. (1998)

## Abstract
This notebook provides a from-scratch PyTorch implementation of the foundational LeNet-5 Convolutional Neural Network (CNN). The objective is to validate the original architectural design by training the model on the MNIST dataset to achieve optimal classification accuracy.

## Mathematical Formulation
The network acts as a multi-layer function mapping an input image $X$ to a probability distribution over 10 classes. The model is optimized using the Cross-Entropy Loss function:

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{c=1}^{C} y_{i,c} \log(\hat{y}_{i,c})$$

Where:
* $N$ is the batch size.
* $C$ is the number of classes (10).
* $y$ is the ground-truth label (one-hot encoded).
* $\hat{y}$ is the predicted probability (after Softmax).

### 0. Imports and Setup
Load all required libraries. The core model architecture and training routines are
imported from their dedicated modules (`model.py`, `train.py`), keeping this notebook
a clean presentation layer.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Import the modular architecture and training routines
from model import LeNet5
from train import train_model, evaluate_model

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

### 1. Data Loading
Load the MNIST dataset (60,000 train / 10,000 test samples).

**Pre-processing pipeline:**
- `Resize(32, 32)`: MNIST images are 28×28; we resize to 32×32 to match the original LeNet-5 input specification.
- `ToTensor()`: Converts PIL Image (H, W) with pixel range [0, 255] → float Tensor (C, H, W) with range [0.0, 1.0].
- `Normalize(mean=0.1307, std=0.3081)`: Applies z-score normalization using the global mean and standard deviation
  of the MNIST training set, centering the data for faster gradient descent convergence.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),           # Match LeNet-5 original input specification
    transforms.ToTensor(),                 # Pixel values [0,255] → float [0.0,1.0]
    transforms.Normalize((0.1307,), (0.3081,))  # z-score normalisation using MNIST global stats
])

train_set = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Mini-batch size of 64 balances gradient noise vs. memory and is a standard choice
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False)

print(f'Training samples : {len(train_set):,}')
print(f'Test samples     : {len(test_set):,}')

### 2. Model Architecture
The `LeNet5` class (defined in `model.py`) exactly mirrors the 7-layer architecture
described in the 1998 paper. We instantiate it and move it to the available compute device.

| Layer | Type               | Output Shape    |
|-------|--------------------|-----------------|
| C1    | Conv2d(1→6, k=5)   | (6, 28, 28)     |
| S2    | AvgPool2d(k=2,s=2) | (6, 14, 14)     |
| C3    | Conv2d(6→16, k=5)  | (16, 10, 10)    |
| S4    | AvgPool2d(k=2,s=2) | (16, 5, 5)      |
| C5    | Linear(400→120)    | (120,)          |
| F6    | Linear(120→84)     | (84,)           |
| Out   | Linear(84→10)      | (10,)           |

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = LeNet5().to(device)

# Cross-Entropy Loss: combines log-softmax and NLL loss for numerical stability
criterion = nn.CrossEntropyLoss()

# Adam optimizer: adaptive learning rate. lr=0.001 is the canonical default.
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal trainable parameters: {total_params:,)')

### 3. Training Loop
We delegate the full training routine to `train_model()` from `train.py`.
Each epoch performs a complete forward and backward pass over all 60,000 training images.

In [ ]:
EPOCHS = 10

losses = train_model(
    model        = model,
    train_loader = train_loader,
    criterion    = criterion,
    optimizer    = optimizer,
    epochs       = EPOCHS,
    device       = device
)

#### Training Loss Curve
The loss curve illustrates the convergence behaviour. A steep initial drop indicates
that the network rapidly learns the most discriminative features in the first few epochs.

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(range(1, EPOCHS + 1), losses, marker='o', linewidth=2, color='steelblue')
plt.title('LeNet-5 Training Loss over Epochs', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.xticks(range(1, EPOCHS + 1))
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('assets/loss_curve.png', dpi=150)
plt.show()

### 2b. Model Checkpointing
Save the trained model's weights (state dictionary) to disk. This allows the model
to be reloaded instantly for inference without retraining, following best practices
in reproducible ML research.

```python
# To reload later:
model = LeNet5()
model.load_state_dict(torch.load('lenet5_mnist.pth'))
model.eval()
```

In [ ]:
# Save only the state_dict (weights), not the full model object — more portable
torch.save(model.state_dict(), 'lenet5_mnist.pth')
print('Model weights saved to lenet5_mnist.pth')

### 4. Evaluation and Validation
Evaluate the trained model on the 10,000 held-out test images using `evaluate_model()`
from `train.py`. No gradients are computed — this is pure inference.

In [ ]:
accuracy, all_preds, all_labels = evaluate_model(model, test_loader, device)
print(f'Test Accuracy: {accuracy:.2f}%')

### 5. Confusion Matrix
The confusion matrix C is a 10×10 grid where entry C[i, j] counts how many images
with **true label** $i$ were **predicted** as label $j$.

* The diagonal represents correct predictions.
* Off-diagonal entries reveal systematic confusions (e.g., '4' mistaken for '9').

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=range(10), yticklabels=range(10)
)
plt.title('Confusion Matrix: LeNet-5 on MNIST Test Set', fontsize=14)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('assets/confusion_matrix.png', dpi=150)
plt.show()

### 6. Feature Map Visualization (Interpretability)
After the C1 convolutional layer, the 6 learned filters transform the input image into
6 feature maps. Each map highlights a different spatial structure the filter is most
sensitive to (e.g., horizontal edges, vertical edges, curves).

This visualization provides empirical evidence that the network is behaving as expected
— learning meaningful low-level feature detectors rather than memorizing noise.

In [ ]:
model.eval()

# Select a single test image to inspect
sample_image, sample_label = test_set[0]
sample_tensor = sample_image.unsqueeze(0).to(device)  # Shape: (1, 1, 32, 32)

# Extract feature maps from the C1 layer using the dedicated method in model.py
with torch.no_grad():
    feature_maps = model.get_feature_maps(sample_tensor)  # Shape: (1, 6, 28, 28)

feature_maps = feature_maps.squeeze(0).cpu().numpy()  # Shape: (6, 28, 28)

fig, axes = plt.subplots(1, 7, figsize=(14, 3))

# Plot the original input image
axes[0].imshow(sample_image.squeeze(), cmap='gray')
axes[0].set_title(f'Input\n(True: {sample_label})', fontsize=9)
axes[0].axis('off')

# Plot each of the 6 C1 feature maps
for i, ax in enumerate(axes[1:]):
    ax.imshow(feature_maps[i], cmap='viridis')
    ax.set_title(f'Filter {i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle('C1 Layer: Feature Maps After First Convolutional Layer', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('assets/feature_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature map visualization saved to assets/feature_maps.png')

### 7. Misclassified Images Analysis
To understand the **1.22% error rate**, we display actual images the model got wrong,
pairing each with its True Label vs. the model's Predicted Label.

Misclassified images often represent handwriting ambiguities that even a human might
find difficult to classify — validating the model's overall robustness.

In [ ]:
# Collect misclassified images from the test set
misclassified_images  = []
misclassified_true    = []
misclassified_pred    = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs        = model(images)
        _, predicted   = torch.max(outputs, dim=1)

        # Find indices where prediction does NOT match ground-truth
        wrong_mask = (predicted != labels)
        misclassified_images.extend(images[wrong_mask].cpu())
        misclassified_true.extend(labels[wrong_mask].cpu().numpy())
        misclassified_pred.extend(predicted[wrong_mask].cpu().numpy())

        if len(misclassified_images) >= 9:
            break

print(f'Total misclassified images collected: {len(misclassified_images)}')

# Plot a 3x3 grid of the first 9 misclassified examples
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
fig.suptitle('Misclassified Images: True Label vs. Predicted Label', fontsize=13)

for idx, ax in enumerate(axes.flat):
    img = misclassified_images[idx].squeeze().numpy()
    ax.imshow(img, cmap='gray')
    ax.set_title(f'True: {misclassified_true[idx]}  |  Pred: {misclassified_pred[idx]}',
                 fontsize=10, color='red')
    ax.axis('off')

plt.tight_layout()
plt.savefig('assets/misclassified.png', dpi=150)
plt.show()
print('Misclassified grid saved to assets/misclassified.png')